In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
csv_path = Path(
    r"D:\Priyanka\5th project\Clinical-trial-disease-classification\data\clinical_trials_raw_patient2trial_conditions.csv"
)

print("File exists:", csv_path.exists())

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())



In [ ]:
print("Rows and columns:", df.shape)

print("\nMissing values per column:")
print(df[["brief_summary", "source_condition_query"]].isna().sum())

print("\nDisease category counts:")
print(df["source_condition_query"].value_counts())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
import matplotlib.pyplot as plt

category_counts = df["source_condition_query"].value_counts()

plt.figure(figsize=(10,6))
category_counts.sort_values().plot(kind="barh",color="steelblue")
plt.title("Clinical Trials by Disease Category")
plt.xlabel("Number of Clinical Trials")
plt.ylabel("Disease Category")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split 
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

x = df["brief_summary"].astype(str)
y = df["source_condition_query"].astype(str)

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(x_train))
print("Testing rows:", len(x_test))

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1,2),
        max_features=50000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
    ))
])

model.fit(x_train, y_train)
print("Model training complete.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(x_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=45,
    cmap="Blues",
    values_format="d"
)

plt.title("Confusion Matrix: Disease Category Classification")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import joblib

In [ ]:
model_folder = Path("models")
model_folder.mkdir(exist_ok=True)

model_path = model_folder / "disease_classification.joblib"
joblib.dump(model,model_path)

print("Model saved to:", model_path.resolve())
